In [ ]:
import json
import re
from urllib import error, request

import openai
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)

client = openai.OpenAI()
API_BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"
DEFAULT_HEADERS = {
    "accept": "application/json",
    "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
}
MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """
너는 영화 질문을 해결하는 Movie Agent다.
항상 한국어로 답하고, 필요한 경우에만 도구를 호출하라.
도구 결과에 포함된 사실만 사용하고, 정보가 부족하면 솔직하게 말하라.
사용자의 이전 대화 맥락을 계속 기억하고 이어서 답하라.
영화 추천, 영화 상세 정보, 비슷한 영화 추천 요청을 해결하기 위해 여러 도구를 연속으로 호출해도 된다.
답변에 이미지, 포스터, 마크다운 이미지 문법(![](...))을 포함하지 마라. 이미지 URL도 본문에 출력하지 마라.
응답은 항상 자연어로 작성하고 JSON, dict, raw payload를 사용자에게 그대로 출력하지 마라.
도구를 호출했다면 최종 답변에서 필요한 내용만 정리해서 설명하라.
출력은 간결하게 작성하라. 불필요하게 길게 설명하지 마라.
마크다운 제목, 굵게, 표를 사용하지 마라.
항상 정보를 잘라내듯 나열하지 말고, 핵심만 짧게 요약해서 말하라.
인기 영화 목록을 보여줄 때는 '현재 인기 영화 목록입니다:' 다음 줄부터 '1. 제목 (ID: 123) - 한줄 요약' 형식으로 최대 5개까지만 보여줘라.
각 영화의 한줄 요약은 장르, 분위기, 특징 중 핵심 1가지만 반영해 짧게 써라.
영화 상세 정보를 설명할 때는 첫 문장에서 'OOO는 ... 영화입니다.'처럼 핵심 소개를 하고, 이어서 1~2문장 안에서 줄거리, 장르, 특징을 자연스럽게 요약하라.
영화 상세 정보에는 긴 줄거리 전체, 과도한 숫자 나열, 장문 불릿 목록을 쓰지 마라.
비슷한 영화 추천을 할 때는 'OOO를 좋아하셨다면 이런 영화를 추천드립니다:' 다음 줄부터 '1. 제목 - 짧은 추천 이유' 형식으로 최대 5개까지 보여줘라.
비슷한 영화 추천의 이유는 작품별로 한 문장보다 짧게, 분위기나 공통점 위주로 요약하라.
항상 예시처럼 대화형으로 자연스럽게 답하되, 군더더기 마무리 문장은 최소화하라.
""".strip()

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    }
]


def request_movie_api(path: str):
    url = f"{API_BASE_URL}{path}"
    api_request = request.Request(url, headers=DEFAULT_HEADERS)
    try:
        with request.urlopen(api_request) as response:
            payload = response.read().decode("utf-8")
        return json.loads(payload)
    except error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        return {
            "error": f"HTTP {exc.code}",
            "path": path,
            "detail": detail,
        }
    except error.URLError as exc:
        return {
            "error": "NETWORK_ERROR",
            "path": path,
            "detail": str(exc.reason),
        }


def get_popular_movies():
    return request_movie_api("/movies")


def get_movie_details(id: int):
    return request_movie_api(f"/movies/{id}")


def get_similar_movies(id: int):
    return request_movie_api(f"/movies/{id}/similar")


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}


TOOLS = [
    {
        "type": "function",
        "function": {
            
            "name": "get_popular_movies",
            "description": "현재 인기 영화 목록이 필요할 때 /movies 엔드포인트를 호출한다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 ID의 상세 정보가 필요할 때 /movies/:id 엔드포인트를 호출한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "조회할 영화의 ID",
                    }
                },
                "required": ["id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화와 비슷한 영화를 찾을 때 /movies/:id/similar 엔드포인트를 호출한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "기준이 되는 영화의 ID",
                    }
                },
                "required": ["id"],
                "additionalProperties": False,
            },
        },
    },
]


In [ ]:
from openai.types.chat import ChatCompletionMessage


def coerce_content(content):
    if content is None:
        return ""
    if isinstance(content, (str, list)):
        return content
    return json.dumps(content, ensure_ascii=False)


def sanitize_message(message):
    item = dict(message)
    item["content"] = coerce_content(item.get("content"))
    return item


def build_safe_messages(history):
    return [sanitize_message(message) for message in history]


def append_assistant_tool_calls(message: ChatCompletionMessage):
    messages.append(
        {
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
                for tool_call in message.tool_calls
            ],
        }
    )


def format_tool_arguments(arguments):
    if not arguments:
        return ""
    return ", ".join(str(value) for value in arguments.values())


def parse_tool_arguments(raw_arguments):
    if not raw_arguments:
        return {}

    try:
        arguments = json.loads(raw_arguments)
    except json.JSONDecodeError:
        return {}

    return arguments if isinstance(arguments, dict) else {}


def append_tool_result(tool_call_id, function_name, result):
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "name": function_name,
            "content": json.dumps(result, ensure_ascii=False),
        }
    )


def run_tool_call(tool_call):
    function_name = tool_call.function.name
    arguments = parse_tool_arguments(tool_call.function.arguments)

    function_to_run = FUNCTION_MAP.get(function_name)
    if function_to_run is None:
        result = {"error": f"Unknown function requested: {function_name}"}
    else:
        try:
            result = function_to_run(**arguments)
        except TypeError as exc:
            result = {
                "error": "INVALID_ARGUMENTS",
                "function": function_name,
                "detail": str(exc),
                "arguments": arguments,
            }

    formatted_arguments = format_tool_arguments(arguments)
    print(f"Agent: [{function_name}({formatted_arguments}) 호출]")

    append_tool_result(tool_call.id, function_name, result)


def strip_image_markdown(text: str) -> str:
    text = re.sub(r"!\[[^\]]*\]\([^\)]+\)", "", text)
    text = text.replace("**", "")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        append_assistant_tool_calls(message)
        for tool_call in message.tool_calls:
            run_tool_call(tool_call)
        return False

    final_text = strip_image_markdown(message.content or "답변을 생성하지 못했습니다.")
    messages.append({"role": "assistant", "content": final_text})
    print(f"Agent: {final_text}")
    return True


def call_ai():
    while True:
        safe_messages = build_safe_messages(messages)
        response = client.chat.completions.create(
            model=MODEL,
            messages=safe_messages,
            tools=TOOLS,
            tool_choice="auto",
        )

        if process_ai_response(response.choices[0].message):
            break


In [ ]:
while True:
    user_message = input("영화 질문을 입력하세요 (quit/q 종료): ")
    if user_message in {"quit", "q"}:
        break

    messages.append({"role": "user", "content": user_message})
    print(f"User: {user_message}")
    call_ai()
